In [1]:
import random
import numpy as np
import os
import torch 

def set_seed(seed=24):
    """Setea semilla para reproducibilidad general"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    
    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # si usas multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"Semilla fijada en: {seed}")

# Llamar a la función
set_seed(24)

OSError: [WinError 182] El sistema operativo no puede ejecutar %1. Error loading "c:\Users\matia\anaconda3\envs\ldi2_cuda\Lib\site-packages\torch\lib\fbgemm.dll" or one of its dependencies.

In [2]:
import sys
sys.path.append('../')
 
import pandas as pd 
from sklearn.metrics import cohen_kappa_score, accuracy_score,balanced_accuracy_score
from plotly import express as px
from tutoriales.utils import plot_confusion_matrix, get_artifact_filename
from json import loads
from joblib import load, dump
import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact
from optuna.visualization import plot_param_importances
from optuna.importance import FanovaImportanceEvaluator
from optuna.importance import MeanDecreaseImpurityImportanceEvaluator  
from optuna.importance import PedAnovaImportanceEvaluator
from optuna.visualization import plot_contour

d:\anaconda3\envs\ldi2_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys, numpy.core
# Shim: permite cargar joblib guardados con numpy >= 2.0 en entornos con numpy 1.x
sys.modules.setdefault("numpy._core", numpy.core)
for _sub in ["numeric", "multiarray", "umath", "fromnumeric", "arrayprint", "strings"]:
    mod = getattr(numpy.core, _sub, numpy.core)
    sys.modules.setdefault(f"numpy._core.{_sub}", mod)

C:\Users\matia\AppData\Local\Temp\ipykernel_11336\2672558879.py:5: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric.
  mod = getattr(numpy.core, _sub, numpy.core)
C:\Users\matia\AppData\Local\Temp\ipykernel_11336\2672558879.py:5: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public

In [4]:
# Paths
BASE_DIR = '../'
PATH_TO_MODELS = os.path.join(BASE_DIR, "work/models")
PATH_TO_TRAIN = os.path.join(BASE_DIR, "work/cleaned/train_clean.csv")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

In [7]:
# Las predicciones del modelo LGB se guardan directamente como joblib
#lgb_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_modelo_1__variables_originales.joblib'))
#lgb_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_modelo_2__variables_completas.joblib'))
lgb_dataset = load(os.path.join(PATH_TO_MODELS, 'stacking_modelos_v1.joblib'))




In [8]:
lgb_dataset

,PetID,pred,AdoptionSpeed
0,8f20e24ef,"[0.010011534178705381, 0.13775704018798024, 0....",4
1,2d72ef0c4,"[0.02856262987913568, 0.18595500709547222, 0.2...",4
2,44cd12263,"[0.016374875943807574, 0.12184898956619723, 0....",4
3,210c4a637,"[0.017225928405245425, 0.2761924178711388, 0.4...",2
4,21493e6ea,"[0.01796733349483558, 0.11559925581605356, 0.2...",4
...,...,...,...
2994,35f9818a7,"[0.014374310868143645, 0.07503438874988333, 0....",4
2995,46e25aa2b,"[0.024738549482908104, 0.1922261477408506, 0.2...",1
2996,d3692d2b2,"[0.049731424977735965, 0.410418347370511, 0.27...",1
2997,3c43b7541,"[0.026574476871864487, 0.14080259644606502, 0....",4


In [93]:
MODEL_NAME = '01 DistilBert'
MODEL_VERSION = '5.0'

study_bert = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3", 
                            study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                            load_if_exists = True)


[I 2026-05-09 01:20:15,792] Using an existing study with name '01 DistilBert_5.0' instead of creating a new one.


In [94]:
bert_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_bert,'test')))

In [95]:
# Cargar el modelo ResNet
MODEL_NAME_RESNET = '04 ResNet Augment'
MODEL_VERSION_RESNET = '1.0.0'

study_resnet = optuna.create_study(
    direction='maximize',
    storage="sqlite:///../work/optuna_artifacts/db.sqlite3",
    study_name=f'{MODEL_NAME_RESNET}_{MODEL_VERSION_RESNET}',
    load_if_exists=True
)

[I 2026-05-09 01:20:15,858] Using an existing study with name '04 ResNet Augment_1.0.0' instead of creating a new one.


In [96]:
resnet_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_04 ResNet Augment_1.0.0_1.joblib'))
#resnet_dataset = load(os.path.join(PATH_TO_TEMP_FILES,get_artifact_filename(study_resnet,'test')))

In [97]:
merged_datasets = lgb_dataset[['PetID', 'pred', 'AdoptionSpeed']].rename({'pred':'lgb_pred_score'},axis=1).merge(bert_dataset[['PetID', 'pred']].rename({'pred':'bert_pred_score'},axis=1),
                  on='PetID', how='outer')

In [98]:
# Unir ResNet al dataframe fusionado
merged_datasets = merged_datasets.merge(
    resnet_dataset[['PetID', 'pred']].rename({'pred': 'resnet_pred_score'}, axis=1),
    on='PetID', how='outer'
)

In [99]:
merged_datasets.head()

,PetID,lgb_pred_score,AdoptionSpeed,bert_pred_score,resnet_pred_score
0,002230dea,"[0.06175187846442381, 0.2940676868352113, 0.41...",1,"[0.009752763, 0.5465382, 0.43560302, 0.0048021...","[-0.99180895, 0.7428905, 0.48204118, 0.1142346..."
1,0063f83c9,"[0.08883536737434454, 0.16060991232554725, 0.2...",1,"[0.0020303065, 0.00568232, 0.088742584, 0.0133...","[-1.3278729, 0.98171896, -0.028774094, 0.35766..."
2,0073c33d0,"[0.006477767488363635, 0.222578069511566, 0.22...",3,"[0.0007240717, 0.016772836, 0.22749364, 0.7542...","[-1.4208313, 0.7716534, 1.2558718, 0.40233982,..."
3,00bfa5da9,"[0.008940191888288453, 0.09638043680171315, 0....",4,"[8.41588e-05, 0.00040059086, 0.0027940436, 0.0...","[-2.9708593, -0.9604822, 0.53046876, 0.9579238..."
4,00c19f4fa,"[0.020924012275847437, 0.12606051674692606, 0....",2,"[0.0014462878, 0.08401318, 0.8484335, 0.056610...","[-2.509712, 0.45661384, 0.80324507, 0.8857457,..."


In [100]:
merged_datasets.isnull().mean().mul(100).round(2).rename('% nulos')


PetID                0.00
lgb_pred_score       0.00
AdoptionSpeed        0.00
bert_pred_score      0.10
resnet_pred_score    2.27
Name: % nulos, dtype: float64

In [101]:
# Limpiar nulos (rellenar con arrays de ceros si algún modelo no tiene predicción para un PetID)
merged_datasets['resnet_pred_score'] = [np.zeros(5) if type(i) is float else i for i in merged_datasets['resnet_pred_score']]
merged_datasets['bert_pred_score']   = [np.zeros(5) if type(i) is float else i for i in merged_datasets['bert_pred_score']]
merged_datasets['lgb_pred_score']    = [np.zeros(5) if type(i) is float else i for i in merged_datasets['lgb_pred_score']]


In [102]:
all_values = [item for sublist in merged_datasets["resnet_pred_score"] for item in sublist]
min_val = min(all_values)
max_val = max(all_values)

def normalizar_lista(lista):
    return [(x - min_val) / (max_val - min_val) for x in lista]

merged_datasets["resnet_pred_score"] = merged_datasets["resnet_pred_score"].apply(normalizar_lista)

all_values1 = [item for sublist in merged_datasets["bert_pred_score"] for item in sublist]
min_val1 = min(all_values1)
max_val1 = max(all_values1)

def normalizar_lista1(lista):
    return [(x - min_val1) / (max_val1 - min_val1) for x in lista]

merged_datasets["bert_pred_score"] = merged_datasets["bert_pred_score"].apply(normalizar_lista1)

all_values2 = [item for sublist in merged_datasets["lgb_pred_score"] for item in sublist]
min_val2 = min(all_values2)
max_val2 = max(all_values2)

def normalizar_lista2(lista):
    return [(x - min_val2) / (max_val2 - min_val2) for x in lista]

merged_datasets["lgb_pred_score"] = merged_datasets["lgb_pred_score"].apply(normalizar_lista2)


In [103]:
merged_datasets.head()

,PetID,lgb_pred_score,AdoptionSpeed,bert_pred_score,resnet_pred_score
0,002230dea,"[0.06338951295469104, 0.3074418540495358, 0.43...",1,"[0.009764634848057976, 0.5472034399846415, 0.4...","[0.3858939, 0.60110545, 0.5687438, 0.5231127, ..."
1,0063f83c9,"[0.09184125250627968, 0.16724182833967538, 0.2...",1,"[0.0020327778574799906, 0.005689236591533274, ...","[0.3442009, 0.6307352, 0.5053706, 0.55331373, ..."
2,0073c33d0,"[0.00532296862330662, 0.2323406003115052, 0.23...",3,"[0.0007249530609934081, 0.016793252943771726, ...","[0.33266822, 0.60467386, 0.66474736, 0.5588558..."
3,00bfa5da9,"[0.007909794047651216, 0.09976749683027117, 0....",4,"[8.426123820892333e-05, 0.00040107846939244556...","[0.14036748, 0.3897804, 0.57475185, 0.6277831,..."
4,00c19f4fa,"[0.020499033847363714, 0.13094700650423355, 0....",2,"[0.001448048319537855, 0.08411544279376652, 0....","[0.19757868, 0.56558925, 0.6085932, 0.6188285,..."


In [104]:
# Optimización con Optuna para 3 pesos
def objective(trial):
    # Definir pesos para los tres modelos
    w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_bert = trial.suggest_float('w_bert', 0.0, 1.0)
    w_resnet = trial.suggest_float('w_resnet', 0.0, 1.0)
    
    # Normalización
    total_w = w_lgb + w_bert + w_resnet
    if total_w == 0: return 0
    
    # Cálculo vectorizado para mayor velocidad
    # Convertimos las columnas de scores en una matriz 3D o sumamos directamente
    lgb_scores = np.stack(merged_datasets['lgb_pred_score'].values)
    bert_scores = np.stack(merged_datasets['bert_pred_score'].values)
    resnet_scores = np.stack(merged_datasets['resnet_pred_score'].values)
    
    combined_scores = (
        (w_lgb / total_w) * lgb_scores + 
        (w_bert / total_w) * bert_scores + 
        (w_resnet / total_w) * resnet_scores
    )
    
    preds_final = np.argmax(combined_scores, axis=1)
    
    return cohen_kappa_score(merged_datasets['AdoptionSpeed'], preds_final, weights='quadratic')

In [142]:
# Ejecutar el estudio
STORAGE_URL = "sqlite:///../work/db-blend.sqlite3"
study_blend = optuna.create_study(
    direction='maximize',
    storage=STORAGE_URL,
    study_name="Ensemble_LGB_BERT_ResNet",
    load_if_exists=True
)
study_blend.optimize(objective, n_trials=100)

[I 2026-05-09 01:41:57,513] Using an existing study with name 'Ensemble_LGB_BERT_ResNet' instead of creating a new one.
[I 2026-05-09 01:41:57,599] Trial 100 finished with value: 0.4033218389685963 and parameters: {'w_lgb': 0.6019543635454638, 'w_bert': 0.22696774459435964, 'w_resnet': 0.9314329720430734}. Best is trial 97 with value: 0.4055735864499894.
[I 2026-05-09 01:41:57,656] Trial 101 finished with value: 0.4034033340634968 and parameters: {'w_lgb': 0.5129392550913028, 'w_bert': 0.19079685322976747, 'w_resnet': 0.9019434213687163}. Best is trial 97 with value: 0.4055735864499894.
[I 2026-05-09 01:41:57,699] Trial 102 finished with value: 0.40264893940812285 and parameters: {'w_lgb': 0.5211198441818383, 'w_bert': 0.186178755504021, 'w_resnet': 0.8940501531253063}. Best is trial 97 with value: 0.4055735864499894.
[I 2026-05-09 01:41:57,740] Trial 103 finished with value: 0.40109751386989223 and parameters: {'w_lgb': 0.545268549368738, 'w_bert': 0.24579863528031798, 'w_resnet': 0.9

In [143]:
# Resultados
best_params = study_blend.best_params
sum_best_w = sum(best_params.values())

print(f"Mejor Kappa: {study_blend.best_value:.4f}")
print(f"Pesos óptimos: {best_params}")

Mejor Kappa: 0.4074
Pesos óptimos: {'w_lgb': 0.4326067450359527, 'w_bert': 0.25172204852764746, 'w_resnet': 0.841538662390898}


In [144]:
best_kappa = study_blend.best_trial.value
print(f"Mejor puntuación Kappa: {best_kappa}")

Mejor puntuación Kappa: 0.40737235623212464


In [145]:
# Crear la columna de predicción final optimizada

def _prediction_vector(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return np.asarray(value, dtype=float)
    return np.zeros(5, dtype=float)

merged_datasets['blend_pred_score'] = [
    (best_params['w_lgb'] / sum_best_w) * _prediction_vector(r['lgb_pred_score']) +
    (best_params['w_bert'] / sum_best_w) * _prediction_vector(r['bert_pred_score']) +
    (best_params['w_resnet'] / sum_best_w) * _prediction_vector(r['resnet_pred_score'])
    for _, r in merged_datasets.iterrows()
]

In [146]:
#merged_datasets[['lgb_pred_score', 'bert_pred_score', 'resnet_pred_score']]
merged_datasets['blend_pred_score']

0       [0.23240901374717404, 0.5089551292896939, 0.50...
1       [0.21620568136200144, 0.3962140479779647, 0.37...
2       [0.1851002211228577, 0.40212914553847046, 0.46...
3       [0.0796712060307375, 0.24332148194715286, 0.38...
4       [0.11501826475120804, 0.3629328929237755, 0.58...
                              ...                        
2994    [0.15998172502408672, 0.4087164097302859, 0.40...
2995    [0.21896899407796577, 0.3764726480686566, 0.46...
2996    [0.2041827737332116, 0.5038681220519958, 0.507...
2997    [0.19443201561079548, 0.47290348777188795, 0.3...
2998    [0.12454020767719234, 0.3792964437838812, 0.51...
Name: blend_pred_score, Length: 2999, dtype: object

In [147]:
#merged_datasets['blend_pred_score']

In [148]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['lgb_pred_score'].apply(np.argmax), 
                    title = 'LGB Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['lgb_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))

In [149]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['bert_pred_score'].apply(np.argmax), 
                    title = 'Bert Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                   merged_datasets['bert_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))


In [150]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['resnet_pred_score'].apply(np.argmax), 
                    title = 'ResNet Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                   merged_datasets['resnet_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))


In [151]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['blend_pred_score'].apply(np.argmax), 
                    title = 'Blended Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                   merged_datasets['blend_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))


In [152]:
fig = plot_param_importances(study_blend, evaluator= FanovaImportanceEvaluator(seed=24))
fig.update_layout(
    title="Importancia de los Modelos en el Ensamble",
    xaxis_title="Importancia relativa",
    yaxis_title="Modelos",
    template="plotly_white"
)
fig.show()



In [153]:
fig = plot_param_importances(study_blend, evaluator= MeanDecreaseImpurityImportanceEvaluator())
fig.update_layout(
    title="Importancia de los Modelos en el Ensamble",
    xaxis_title="Importancia relativa",
    yaxis_title="Modelos",
    template="plotly_white"
)
fig.show()


In [154]:
#fig = plot_param_importances(study_blend, evaluator= PedAnovaImportanceEvaluator(seed=24))
#fig.update_layout(
#    title="Importancia de los Modelos en el Ensamble",
#    xaxis_title="Importancia relativa",
#    yaxis_title="Modelos",
#    template="plotly_white"
#)
#fig.show()

evaluator = PedAnovaImportanceEvaluator(baseline_quantile=0.25)

# 4. Graficar la importancia
fig = plot_param_importances(
    study_blend,
    evaluator=evaluator
)

# Mostrar la figura
fig.show()

C:\Users\matia\AppData\Local\Temp\ipykernel_7944\1054709255.py:10: ExperimentalWarning: PedAnovaImportanceEvaluator is experimental (supported from v3.6.0). The interface can change in the future.
  evaluator = PedAnovaImportanceEvaluator(baseline_quantile=0.25)
d:\anaconda3\envs\ldi2_cuda\Lib\site-packages\optuna\importance\_ped_anova\evaluator.py:156: UserWarning: `baseline_quantile` has been deprecated in v4.7.0. This feature will be removed in v5.0.0. See https://github.com/optuna/optuna/releases/tag/v4.7.0. `baseline_quantile` is currently ignored. Use `target_quantile` instead.
  optuna_warn(


In [116]:
#fig_contour = plot_contour(study_blend, params=['w_lgb', 'w_bert', 'w_resnet'])

#fig_contour.update_layout(
#    title="Interacción de Pesos y Rendimiento (Kappa)",
#    width=900,
#    height=800
#)

#fig_contour.show()

In [117]:
# Guardar el resultado final en un archivo temporal y subirlo a Optuna
final_filename = "merged_predictions_optimized.joblib"
dump(merged_datasets, final_filename)


['merged_predictions_optimized.joblib']

In [118]:
# Subir el archivo como artefacto del mejor trial
#artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)
#upload_artifact(
#    trial=study_blend.best_trial, 
#    file_path=final_filename, 
#    artifact_store=artifact_store
#)


In [119]:
df1 = study_blend.trials_dataframe()

# El valor de Kappa estará en la columna 'user_attrs_kappa'
#mejor_kappa = df1['user_attrs_kappa'].max() 
#print(f"El mejor Kappa registrado fue: {mejor_kappa}")

In [120]:
df1

,number,value,datetime_start,datetime_complete,duration,params_w_bert,params_w_lgb,params_w_resnet,state
0,0,0.320047,2026-05-09 01:20:16.147692,2026-05-09 01:20:16.185754,0 days 00:00:00.038062,0.027042,0.999551,0.174951,COMPLETE
1,1,0.367802,2026-05-09 01:20:16.201384,2026-05-09 01:20:16.234940,0 days 00:00:00.033556,0.382251,0.584529,0.521884,COMPLETE
2,2,0.349587,2026-05-09 01:20:16.245925,2026-05-09 01:20:16.276047,0 days 00:00:00.030122,0.439314,0.461014,0.737501,COMPLETE
3,3,0.305418,2026-05-09 01:20:16.283012,2026-05-09 01:20:16.317297,0 days 00:00:00.034285,0.455955,0.382022,0.014849,COMPLETE
4,4,0.343846,2026-05-09 01:20:16.326640,2026-05-09 01:20:16.352982,0 days 00:00:00.026342,0.743206,0.775147,0.926616,COMPLETE
...,...,...,...,...,...,...,...,...,...
95,95,0.403956,2026-05-09 01:20:20.393825,2026-05-09 01:20:20.433637,0 days 00:00:00.039812,0.196261,0.514818,0.878575,COMPLETE
96,96,0.403675,2026-05-09 01:20:20.436648,2026-05-09 01:20:20.477026,0 days 00:00:00.040378,0.195354,0.518557,0.892513,COMPLETE
97,97,0.405574,2026-05-09 01:20:20.477026,2026-05-09 01:20:20.524293,0 days 00:00:00.047267,0.192869,0.506942,0.897145,COMPLETE
98,98,0.403504,2026-05-09 01:20:20.535632,2026-05-09 01:20:20.567308,0 days 00:00:00.031676,0.211000,0.578479,0.942214,COMPLETE
